# 01 - Data loading and cleaning
Loads the IBM Telco Customer Churn dataset and applies **rule-based** cleaning only (no statistics are learned here, so nothing can leak from test data).

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
import numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from src import config
config.ensure_directories()

In [2]:
from src.data.load_data import load_raw_data
from src.data.clean_data import clean_raw_dataframe, split_features_target
raw = load_raw_data()
raw.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print(raw.shape)
raw.dtypes

(7043, 21)


customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

## The `TotalCharges` problem
It is stored as text because some cells are blank strings. These belong to customers with `tenure == 0` (not billed yet).

In [4]:
blank = raw[raw['TotalCharges'].astype(str).str.strip() == '']
print(len(blank), 'blank TotalCharges rows')
blank[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

11 blank TotalCharges rows


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [5]:
df, report = clean_raw_dataframe(raw)
report

{'rows_raw': 7043,
 'columns_raw': 21,
 'total_charges_blank_or_invalid': 11,
 'total_charges_set_to_zero_for_tenure_0': 11,
 'exact_duplicate_rows_removed': 0,
 'duplicate_customer_ids': 0,
 'rows_with_identical_features_kept': 40,
 'constant_columns': [],
 'missing_values_after_cleaning': {},
 'rows_clean': 7043,
 'churn_rate': 0.2653698707936959}

In [6]:
print(df.dtypes)
print('Missing values after cleaning:', int(df[config.RAW_FEATURE_COLUMNS].isna().sum().sum()))
print(df[config.TARGET_COLUMN].value_counts(normalize=True).rename({0: 'Not churn', 1: 'Churn'}))

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object
Missing values after cleaning: 0
Churn
Not churn    0.73463
Churn        0.26537
Name: proportion, dtype: float64


## Features / target / identifier
`customerID` is an identifier and is **never** used as a feature. The target `Churn` is encoded Yes -> 1, No -> 0.

In [7]:
X, y, ids = split_features_target(df)
print(X.shape, y.shape)
X.head()

(7043, 19) (7043,)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
